# 03. Интеграция CV, спутниковых и погодных данных

Цель: соединить основную таблицу спутниковых наблюдений и погоды с
результатами CV-классификации цветения. Единица наблюдения — одна
дата и один спутник (`date + sensor`).

CV-поля — это не прогноз на 7 дней. Они описывают текущее состояние
сцены и будут признаками для будущей прогнозной модели.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, balanced_accuracy_score, brier_score_loss, confusion_matrix, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style='whitegrid', context='notebook')
ROOT = Path.cwd()
BASE_FILE = ROOT / 'shershnevskoe_satellite_weather_merged.csv'
CV_FILE = ROOT / 'dataCV' / 'shershnevskoe_cv_all_scenes.csv'
OUTPUT_FILE = ROOT / 'shershnevskoe_satellite_weather_cv_merged.csv'
LABELED_OUTPUT_FILE = ROOT / 'shershnevskoe_satellite_weather_cv_labeled.csv'
BLOOM_PROBABILITY_THRESHOLD = 0.45
TARGET_MIN_DAYS, TARGET_MAX_DAYS = 5, 9

## 1. Загрузка и проверка ключей

In [ ]:
base = pd.read_csv(BASE_FILE, parse_dates=['date'])
cv = pd.read_csv(CV_FILE, parse_dates=['date'])

for name, frame in {'satellite_weather': base, 'cv': cv}.items():
    duplicates = frame.duplicated(['date', 'sensor']).sum()
    print(f'{name}: {len(frame)} rows; duplicate date+sensor: {duplicates}')
    assert duplicates == 0, f'{name} contains duplicate merge keys'

display(base[['date', 'sensor', 'FAI_mean', 'air_temp_c', 'wind_speed_10m_ms']].head())
display(cv[['date', 'sensor', 'bloom_fraction_non_artifact', 'artifact_fraction', 'cv_usable_strict']].head())

## 2. Объединение таблиц

In [ ]:
cv_rename = {
    'artifact_area_m2': 'cv_artifact_area_m2',
    'artifact_fraction': 'cv_artifact_fraction',
    'bloom_area_m2': 'cv_bloom_area_m2',
    'bloom_fraction_all_water': 'cv_bloom_fraction_all_water',
    'bloom_fraction_non_artifact': 'cv_bloom_fraction_non_artifact',
    'classified_water_area_m2': 'cv_classified_water_area_m2',
    'image_id': 'cv_image_id',
    'mean_bloom_probability': 'cv_mean_bloom_probability',
    'mean_fai': 'cv_mean_fai',
    'valid_area_m2': 'cv_valid_area_m2',
    'source_batch': 'cv_source_batch',
    'cv_has_classified_water': 'cv_has_classified_water',
    'cv_usable_strict': 'cv_usable_strict',
    'cv_quality_reason': 'cv_quality_reason',
}
cv_for_merge = cv[['date', 'sensor', *cv_rename]].rename(columns=cv_rename)
cv_for_merge['cv_record_available'] = True

merged = base.merge(cv_for_merge, how='outer', on=['date', 'sensor'], indicator='merge_source')
merged['satellite_weather_record_available'] = merged['merge_source'].ne('right_only')
merged['cv_record_available'] = merged['cv_record_available'].fillna(False).astype(bool)
merged = merged.sort_values(['date', 'sensor']).reset_index(drop=True)

print('Merged shape:', merged.shape)
display(merged['merge_source'].value_counts().rename_axis('source').reset_index(name='rows'))
display(merged.groupby('sensor').size().rename('rows').reset_index())

`both` — строка есть и в старой спутниково-погодной базе, и в CV.
`right_only` — CV-сцена есть, но для неё ещё нет исходной строки с
погодой/агрегированными спутниковыми признаками. Такие строки не
удаляем: они явно помечены и могут быть дополнены позже.

## 3. Контроль качества CV

In [ ]:
quality = (merged.groupby(['sensor', 'cv_quality_reason'], dropna=False)
    .size().rename('rows').reset_index())
display(quality.sort_values(['sensor', 'rows'], ascending=[True, False]))

usable = merged[merged['cv_usable_strict'].fillna(False)].copy()
print(f'Usable CV observations: {len(usable)} of {len(merged)}')
print('Rule: classified water exists, artifact fraction <= 0.30, probability is present.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=merged, x='cv_artifact_fraction', hue='sensor', bins=30,
             stat='density', common_norm=False, ax=axes[0])
axes[0].axvline(0.30, ls='--', color='black', label='quality threshold = 0.30')
axes[0].set(title='Artifact fraction by satellite', xlabel='CV artifact fraction')
axes[0].legend()

sns.histplot(data=usable, x='cv_bloom_fraction_non_artifact', hue='sensor', bins=30,
             stat='density', common_norm=False, ax=axes[1])
axes[1].set(title='Bloom fraction in usable observations', xlabel='Bloom fraction among non-artifact water')
plt.tight_layout()
plt.show()

## 4. Временная динамика CV-сигнала

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
for sensor, group in usable.groupby('sensor'):
    ax.scatter(group['date'], group['cv_mean_bloom_probability'], s=20,
               alpha=0.7, label=sensor)
ax.set(title='Mean CV probability of bloom: usable scenes only',
       xlabel='Date', ylabel='Mean P(bloom)')
ax.legend(title='Sensor')
plt.tight_layout()
plt.show()

## 5. Связь CV-сигнала с FAI и погодой

In [ ]:
analysis = usable.dropna(subset=['FAI_mean', 'air_temp_c', 'cv_mean_bloom_probability']).copy()
print('Rows with CV + FAI + air temperature:', len(analysis))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=analysis, x='FAI_mean', y='cv_mean_bloom_probability',
                hue='sensor', alpha=0.7, ax=axes[0])
axes[0].set(title='CV probability vs satellite FAI')
sns.scatterplot(data=analysis, x='air_temp_c', y='cv_mean_bloom_probability',
                hue='sensor', alpha=0.7, ax=axes[1], legend=False)
axes[1].set(title='CV probability vs air temperature')
plt.tight_layout()
plt.show()

display(analysis[['cv_mean_bloom_probability', 'cv_bloom_fraction_non_artifact',
                  'FAI_mean', 'air_temp_c', 'wind_speed_10m_ms', 'solar_radiation_mj_m2']]
        .corr(numeric_only=True).round(2))

## 6. Построение целевой переменной без смешивания FAI спутников

In [ ]:
# We use a CV-based semantic label, not a raw FAI threshold.
# Raw FAI values differ between Landsat and Sentinel-2.
manual = pd.read_csv(ROOT / 'manual_bloom_validation_template.csv', parse_dates=['date'])
reviewed = (manual.merge(
    merged[['date', 'sensor', 'cv_usable_strict', 'cv_mean_bloom_probability',
            'cv_bloom_fraction_non_artifact']],
    on=['date', 'sensor'], how='left')
    .query("manual_label in ['bloom', 'no_bloom'] and cv_usable_strict == True")
    .copy())
reviewed['manual_bloom'] = (reviewed['manual_label'] == 'bloom').astype(int)

threshold_rows = []
for threshold in np.arange(0.20, 0.76, 0.05):
    prediction = (reviewed['cv_mean_bloom_probability'] >= threshold).astype(int)
    threshold_rows.append({
        'threshold': round(float(threshold), 2),
        'reviewed_scenes': len(reviewed),
        'balanced_accuracy': balanced_accuracy_score(reviewed['manual_bloom'], prediction),
        'f1_bloom': f1_score(reviewed['manual_bloom'], prediction, zero_division=0),
    })
threshold_scores = pd.DataFrame(threshold_rows)
display(threshold_scores.sort_values(['balanced_accuracy', 'f1_bloom'], ascending=False).head(6))
print('Working threshold:', BLOOM_PROBABILITY_THRESHOLD)
print('It is provisional: only manually reviewed, usable scenes are used for this check.')

In [ ]:
labeled = merged.copy()
labeled['season_sin'] = np.sin(2 * np.pi * labeled['date'].dt.dayofyear / 365.25)
labeled['season_cos'] = np.cos(2 * np.pi * labeled['date'].dt.dayofyear / 365.25)

def sensor_month_zscore(values):
    scale = values.std(ddof=0)
    return (values - values.median()) / scale if pd.notna(scale) and scale > 0 else np.nan

# FAI becomes comparable only after normalisation within each
# satellite and calendar month. Raw FAI is not used as a common threshold.
labeled['fai_sensor_month_z'] = labeled.groupby(['sensor', 'month'])['FAI_mean'].transform(sensor_month_zscore)
current_usable = labeled['cv_usable_strict'].fillna(False) & labeled['cv_mean_bloom_probability'].notna()
labeled['bloom_now_cv'] = pd.Series(pd.NA, index=labeled.index, dtype='Int64')
labeled.loc[current_usable, 'bloom_now_cv'] = (
    labeled.loc[current_usable, 'cv_mean_bloom_probability'] >= BLOOM_PROBABILITY_THRESHOLD
).astype(int)

# Target: closest usable CV observation occurring 5–9 days later.
# Cross-sensor matching is allowed because the target is semantic
# (bloom / no bloom), not a raw FAI value.
labeled['bloom_in_7d_cv'] = pd.Series(pd.NA, index=labeled.index, dtype='Int64')
labeled['bloom_target_date_7d'] = pd.NaT
labeled['bloom_target_delta_days'] = np.nan
eligible = labeled.loc[labeled['bloom_now_cv'].notna()].sort_values('date')
for index, row in eligible.iterrows():
    candidates = eligible.loc[
        (eligible['date'] >= row['date'] + pd.Timedelta(days=TARGET_MIN_DAYS)) &
        (eligible['date'] <= row['date'] + pd.Timedelta(days=TARGET_MAX_DAYS))
    ].copy()
    if candidates.empty:
        continue
    candidates['distance_from_7d'] = (candidates['date'] - (row['date'] + pd.Timedelta(days=7))).abs()
    target = candidates.sort_values(['distance_from_7d', 'date']).iloc[0]
    labeled.loc[index, 'bloom_in_7d_cv'] = int(target['bloom_now_cv'])
    labeled.loc[index, 'bloom_target_date_7d'] = target['date']
    labeled.loc[index, 'bloom_target_delta_days'] = (target['date'] - row['date']).days

labeled['bloom_target_status'] = np.where(
    labeled['bloom_now_cv'].isna(), 'current_scene_not_usable',
    np.where(labeled['bloom_in_7d_cv'].isna(), 'no_usable_scene_in_5_9d', 'target_available')
)
display(labeled['bloom_target_status'].value_counts().rename_axis('status').reset_index(name='rows'))
display(labeled.loc[labeled['bloom_in_7d_cv'].notna(),
    ['date', 'sensor', 'bloom_now_cv', 'bloom_target_date_7d',
     'bloom_target_delta_days', 'bloom_in_7d_cv']].head(10))

`bloom_now_cv` — текущая метка 0/1 только для качественных сцен.
`bloom_in_7d_cv` — целевая переменная прогноза. Пустое значение
означает «нет качественного наблюдения для метки», а не «цветения нет».

## 7. Первый прогнозный классификатор: временной holdout

In [ ]:
forecast = labeled.dropna(subset=['bloom_in_7d_cv']).copy()
forecast['bloom_in_7d_cv'] = forecast['bloom_in_7d_cv'].astype(int)
print('Forecast rows:', len(forecast))
display(forecast.groupby(forecast['date'].dt.year)['bloom_in_7d_cv']
        .agg(rows='size', blooms='sum').reset_index(names='year'))

feature_columns = [
    'cv_mean_bloom_probability', 'cv_bloom_fraction_non_artifact', 'cv_artifact_fraction',
    'fai_sensor_month_z',
    'air_temp_c', 'wind_speed_10m_ms', 'precipitation_mm', 'solar_radiation_mj_m2',
    'air_temp_mean_prev_7d', 'lake_temp_mean_prev_7d', 'precipitation_sum_prev_7d',
    'wind_mean_prev_7d', 'calm_days_prev_7d', 'solar_sum_prev_7d',
    'season_sin', 'season_cos'
]
categorical_columns = ['sensor']
train = forecast[forecast['date'].dt.year <= 2023].copy()
test = forecast[forecast['date'].dt.year >= 2024].copy()
print(f'Train: {len(train)} rows, blooms={train.bloom_in_7d_cv.sum()}')
print(f'Test: {len(test)} rows, blooms={test.bloom_in_7d_cv.sum()}')

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), feature_columns),
    ('sensor', OneHotEncoder(handle_unknown='ignore'), categorical_columns),
])

def metrics_row(name, y_true, probability):
    predicted = (probability >= 0.5).astype(int)
    return {
        'model': name,
        'accuracy': accuracy_score(y_true, predicted),
        'balanced_accuracy': balanced_accuracy_score(y_true, predicted),
        'f1_bloom': f1_score(y_true, predicted, zero_division=0),
        'roc_auc': roc_auc_score(y_true, probability) if y_true.nunique() == 2 else np.nan,
        'confusion_matrix_[tn,fp;fn,tp]': confusion_matrix(y_true, predicted).tolist(),
    }

In [ ]:
X_train, y_train = train[feature_columns + categorical_columns], train['bloom_in_7d_cv']
X_test, y_test = test[feature_columns + categorical_columns], test['bloom_in_7d_cv']

# Persistence baseline: current CV label is used as a seven-day forecast.
baseline_probability = test['bloom_now_cv'].astype(float).to_numpy()
scores = [metrics_row('persistence baseline', y_test, baseline_probability)]

models = {
    'logistic regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'random forest': RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_leaf=3,
        class_weight='balanced_subsample', random_state=42, n_jobs=-1
    ),
}
fitted_models = {}
for name, estimator in models.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipeline.fit(X_train, y_train)
    probability = pipeline.predict_proba(X_test)[:, 1]
    scores.append(metrics_row(name, y_test, probability))
    fitted_models[name] = (pipeline, probability)

score_table = pd.DataFrame(scores).set_index('model').round(3)
display(score_table)

In [ ]:
prediction_view = test[['date', 'sensor', 'bloom_now_cv', 'bloom_target_date_7d',
                        'bloom_target_delta_days', 'bloom_in_7d_cv']].copy()
for name, (_, probability) in fitted_models.items():
    prediction_view['p_' + name.replace(' ', '_')] = probability
display(prediction_view.sort_values('date'))

print('Interpretation: this is a feasibility prototype, not a final forecast model.')
print('The temporal test is intentionally late (2024+); it prevents random mixing of future and past dates.')
print('With only this many 7-day targets, metric changes should not be treated as stable evidence.')

## 8. Relaxed-режим: порог артефактов 0.50

In [ ]:
RELAXED_ARTIFACT_THRESHOLD = 0.50
labeled['cv_usable_relaxed_050'] = (
    labeled['cv_classified_water_area_m2'].gt(0) &
    labeled['cv_artifact_fraction'].le(RELAXED_ARTIFACT_THRESHOLD) &
    labeled['cv_mean_bloom_probability'].notna()
)
labeled['bloom_now_cv_050'] = pd.Series(pd.NA, index=labeled.index, dtype='Int64')
labeled.loc[labeled['cv_usable_relaxed_050'], 'bloom_now_cv_050'] = (
    labeled.loc[labeled['cv_usable_relaxed_050'], 'cv_mean_bloom_probability'] >= BLOOM_PROBABILITY_THRESHOLD
).astype(int)
labeled['bloom_in_7d_cv_050'] = pd.Series(pd.NA, index=labeled.index, dtype='Int64')
labeled['bloom_target_date_7d_050'] = pd.NaT
labeled['bloom_target_delta_days_050'] = np.nan
relaxed_eligible = labeled[labeled['bloom_now_cv_050'].notna()].sort_values('date')
for index, row in relaxed_eligible.iterrows():
    candidates = relaxed_eligible.loc[
        (relaxed_eligible['date'] >= row['date'] + pd.Timedelta(days=TARGET_MIN_DAYS)) &
        (relaxed_eligible['date'] <= row['date'] + pd.Timedelta(days=TARGET_MAX_DAYS))
    ].copy()
    if candidates.empty:
        continue
    candidates['distance_from_7d'] = (candidates['date'] - (row['date'] + pd.Timedelta(days=7))).abs()
    target = candidates.sort_values(['distance_from_7d', 'date']).iloc[0]
    labeled.loc[index, 'bloom_in_7d_cv_050'] = int(target['bloom_now_cv_050'])
    labeled.loc[index, 'bloom_target_date_7d_050'] = target['date']
    labeled.loc[index, 'bloom_target_delta_days_050'] = (target['date'] - row['date']).days

labeled['bloom_target_status_050'] = np.where(
    labeled['bloom_now_cv_050'].isna(), 'current_scene_not_usable',
    np.where(labeled['bloom_in_7d_cv_050'].isna(), 'no_usable_scene_in_5_9d', 'target_available')
)

comparison = pd.DataFrame([
    {'quality_mode': 'strict: artifacts <= 0.30',
     'current_usable_scenes': labeled['bloom_now_cv'].notna().sum(),
     'available_7d_targets': labeled['bloom_in_7d_cv'].notna().sum()},
    {'quality_mode': 'relaxed: artifacts <= 0.50',
     'current_usable_scenes': labeled['bloom_now_cv_050'].notna().sum(),
     'available_7d_targets': labeled['bloom_in_7d_cv_050'].notna().sum()},
])
display(comparison)

Relaxed-режим увеличивает выборку, но не отменяет строгий фильтр.
`cv_usable_strict` остаётся эталонным флагом качества, а 0.50 используется
только для исследования вероятностной модели и чувствительности результата.

## 9. Вероятностный прогноз через ~7 дней на relaxed-выборке

In [ ]:
forecast_050 = labeled.dropna(subset=['bloom_in_7d_cv_050']).copy()
forecast_050['bloom_in_7d_cv_050'] = forecast_050['bloom_in_7d_cv_050'].astype(int)
train_050 = forecast_050[forecast_050['date'].dt.year <= 2023].copy()
test_050 = forecast_050[forecast_050['date'].dt.year >= 2024].copy()
print(f'Relaxed target rows: {len(forecast_050)}, blooms={forecast_050.bloom_in_7d_cv_050.sum()}')
print(f'Train: {len(train_050)}, blooms={train_050.bloom_in_7d_cv_050.sum()}')
print(f'Test: {len(test_050)}, blooms={test_050.bloom_in_7d_cv_050.sum()}')

X_train_050 = train_050[feature_columns + categorical_columns]
y_train_050 = train_050['bloom_in_7d_cv_050']
X_test_050 = test_050[feature_columns + categorical_columns]
y_test_050 = test_050['bloom_in_7d_cv_050']

probability_models = {
    # This is the operationally simplest probability: P(current bloom)
    # used as P(bloom persists for roughly one week).
    'persistence: current CV probability': test_050['cv_mean_bloom_probability'].to_numpy(),
}
probability_estimators = {
    'logistic regression probability': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'random forest probability': RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_leaf=3,
        class_weight='balanced_subsample', random_state=42, n_jobs=-1
    ),
}
fitted_probability_models = {}
for name, estimator in probability_estimators.items():
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', estimator)])
    pipeline.fit(X_train_050, y_train_050)
    probability_models[name] = pipeline.predict_proba(X_test_050)[:, 1]
    fitted_probability_models[name] = pipeline

probability_scores = []
for name, probability in probability_models.items():
    probability_scores.append({
        'model': name,
        'roc_auc': roc_auc_score(y_test_050, probability),
        # Lower Brier score means better calibrated probabilities.
        'brier_score': brier_score_loss(y_test_050, probability),
        'mean_predicted_probability': np.mean(probability),
    })
display(pd.DataFrame(probability_scores).set_index('model').round(3))

In [ ]:
# No 0.5 classification threshold is applied below. These are probabilities.
probability_output = test_050[['date', 'sensor', 'bloom_target_date_7d_050',
                               'bloom_target_delta_days_050', 'bloom_in_7d_cv_050']].copy()
for name, probability in probability_models.items():
    probability_output['p_' + name.replace(': ', '_').replace(' ', '_')] = probability
display(probability_output.sort_values('date'))

PROBABILITY_OUTPUT_FILE = ROOT / 'shershnevskoe_7d_probability_holdout_relaxed_050.csv'
probability_save = probability_output.copy()
probability_save['date'] = probability_save['date'].dt.strftime('%Y-%m-%d')
probability_save['bloom_target_date_7d_050'] = probability_save['bloom_target_date_7d_050'].dt.strftime('%Y-%m-%d')
probability_save.to_csv(PROBABILITY_OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Saved probability holdout: {PROBABILITY_OUTPUT_FILE.resolve()}')
print('Caution: only one positive target is present in this late temporal test.')

## 10. Основная оценка: group-aware stratified cross-validation

В прежнем временном holdout в тестовой части оказалось только одно
цветение, поэтому его метрики нестабильны. Здесь используется
`StratifiedGroupKFold` с 4 фолдами:

- стратификация удерживает в каждом фолде и цветения, и нецветения;
- группировка по `bloom_target_date_7d_050` не позволяет двум строкам,
  которые ссылаются на одну будущую сцену-метку, попасть одновременно
  в train и test;
- это честнее для сравнения моделей на малой выборке, но **не** заменяет
  поздний временной тест: годы здесь перемешиваются.

Везде ниже модель возвращает `P(цветение примерно через 7 дней)`.
Порог 0.50 используется только в фильтре качества CV-артефактов;
итоговые вероятности не превращаются в 0/1.

In [ ]:
cv_data = labeled.dropna(subset=['bloom_in_7d_cv_050', 'bloom_target_date_7d_050']).copy()
cv_data['bloom_in_7d_cv_050'] = cv_data['bloom_in_7d_cv_050'].astype(int)
cv_data['target_group'] = cv_data['bloom_target_date_7d_050'].dt.strftime('%Y-%m-%d')

y_cv = cv_data['bloom_in_7d_cv_050']
groups_cv = cv_data['target_group']
splitter = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)

fold_rows = []
for fold, (train_index, test_index) in enumerate(splitter.split(cv_data, y_cv, groups_cv), start=1):
    fold_rows.append({
        'fold': fold,
        'train_rows': len(train_index),
        'train_blooms': int(y_cv.iloc[train_index].sum()),
        'test_rows': len(test_index),
        'test_blooms': int(y_cv.iloc[test_index].sum()),
        'test_unique_future_dates': groups_cv.iloc[test_index].nunique(),
    })
fold_summary = pd.DataFrame(fold_rows)
print(f'CV rows: {len(cv_data)}; blooms: {y_cv.sum()}; unique future target dates: {groups_cv.nunique()}')
display(fold_summary)

In [ ]:
def make_preprocessor():
    return ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), feature_columns),
        ('sensor', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_columns),
    ], sparse_threshold=0)

# CatBoost is intentionally not silently installed here. It is absent
# in this environment; HistGradientBoosting is sklearn's modern
# histogram-based tree boosting counterpart.
cv_estimators = {
    'logistic regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'random forest': RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_leaf=3,
        class_weight='balanced_subsample', random_state=42, n_jobs=-1
    ),
    'histogram gradient boosting': HistGradientBoostingClassifier(
        max_iter=100, learning_rate=0.05, max_leaf_nodes=8,
        min_samples_leaf=8, l2_regularization=5.0, random_state=42
    ),
    'classic gradient boosting': GradientBoostingClassifier(
        n_estimators=75, learning_rate=0.03, max_depth=2,
        min_samples_leaf=6, random_state=42
    ),
}

oof = cv_data[['date', 'sensor', 'bloom_target_date_7d_050',
               'bloom_target_delta_days_050', 'bloom_in_7d_cv_050', 'target_group']].copy()
oof['p_persistence_current_cv_probability'] = cv_data['cv_mean_bloom_probability'].to_numpy()
for model_name in cv_estimators:
    oof['p_' + model_name.replace(' ', '_')] = np.nan

for fold, (train_index, test_index) in enumerate(splitter.split(cv_data, y_cv, groups_cv), start=1):
    fold_train = cv_data.iloc[train_index]
    fold_test = cv_data.iloc[test_index]
    X_fold_train = fold_train[feature_columns + categorical_columns]
    X_fold_test = fold_test[feature_columns + categorical_columns]
    y_fold_train = fold_train['bloom_in_7d_cv_050']

    for model_name, estimator in cv_estimators.items():
        model = Pipeline([('preprocessor', make_preprocessor()), ('model', estimator)])
        model.fit(X_fold_train, y_fold_train)
        probability = model.predict_proba(X_fold_test)[:, 1]
        oof.loc[fold_test.index, 'p_' + model_name.replace(' ', '_')] = probability

assert oof.filter(like='p_').notna().all().all(), 'Every row must receive an out-of-fold probability'
print('Each probability is out-of-fold: its row was not used to train that model.')

In [ ]:
probability_columns = [column for column in oof if column.startswith('p_')]
oof_scores = []
for column in probability_columns:
    probability = oof[column]
    oof_scores.append({
        'model': column.removeprefix('p_').replace('_', ' '),
        'roc_auc': roc_auc_score(y_cv, probability),
        'brier_score': brier_score_loss(y_cv, probability),
        'pr_auc': average_precision_score(y_cv, probability),
        'mean_predicted_probability': probability.mean(),
    })
oof_score_table = (pd.DataFrame(oof_scores)
    .sort_values(['brier_score', 'roc_auc'], ascending=[True, False])
    .set_index('model'))
display(oof_score_table.round(3))

print('ROC-AUC and PR-AUC: higher is better. Brier score: lower is better (probabilities closer to observed outcomes).')
print('This compares methods; it is not yet a claim of performance on a wholly unseen future year.')

In [ ]:
# A compact calibration check for the model with the lowest OOF Brier score.
best_probability_column = 'p_' + oof_score_table.index[0].replace(' ', '_')
calibration = oof[['bloom_in_7d_cv_050', best_probability_column]].copy()
calibration['probability_bin'] = pd.cut(
    calibration[best_probability_column], bins=np.linspace(0, 1, 6), include_lowest=True
)
calibration_summary = (calibration.groupby('probability_bin', observed=True)
    .agg(rows=('bloom_in_7d_cv_050', 'size'),
         observed_bloom_rate=('bloom_in_7d_cv_050', 'mean'),
         mean_predicted_probability=(best_probability_column, 'mean'))
    .reset_index())
display(calibration_summary.round(3))

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], '--', color='gray', label='ideal calibration')
ax.scatter(calibration_summary['mean_predicted_probability'],
           calibration_summary['observed_bloom_rate'],
           s=35 + calibration_summary['rows'] * 12,
           label='OOF probability bins')
for _, row in calibration_summary.iterrows():
    ax.annotate(f"n={int(row['rows'])}",
                (row['mean_predicted_probability'], row['observed_bloom_rate']),
                xytext=(4, 4), textcoords='offset points')
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='mean predicted probability',
       ylabel='observed bloom rate', title='OOF calibration: best Brier-score model')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
OOF_PROBABILITY_FILE = ROOT / 'shershnevskoe_7d_probability_oof_relaxed_050.csv'
oof_save = oof.sort_values(['date', 'sensor']).copy()
oof_save['date'] = oof_save['date'].dt.strftime('%Y-%m-%d')
oof_save['bloom_target_date_7d_050'] = oof_save['bloom_target_date_7d_050'].dt.strftime('%Y-%m-%d')
oof_save.to_csv(OOF_PROBABILITY_FILE, index=False, encoding='utf-8-sig')
display(oof.sort_values(['date', 'sensor']).head(10))
print(f'Saved out-of-fold probabilities: {OOF_PROBABILITY_FILE.resolve()}')

## 11. Исследование: прогноз без CV-признаков

Здесь проверяем главный практический вопрос: сможет ли модель
прогнозировать цветение, если на сегодняшнюю дату нет результата
CV-классификации. Для честного сравнения остаются **те же 94 строки,
тот же таргет и те же 4 фолда**, но из входов удаляются все поля,
начинающиеся с `cv_`, и любые текущие CV-метки.

CV всё ещё нужен лишь исторически для построения таргета и отбора
пригодных строк. Входы этой модели — только исходные данные:
спутниковые индексы, качество исходной сцены, погода, погодные
накопления, сезон и тип спутника.

In [ ]:
original_only_features = [
    # Original satellite aggregates, obtained before CV classification.
    'FAI_mean', 'FAI_median', 'FAI_std', 'FAI_p90',
    'MNDWI_mean', 'MNDWI_std', 'MNDWI_p90',
    'Turbidity_FNU_mean', 'Turbidity_FNU_std', 'Turbidity_FNU_p90',
    'SPM_mgL_mean', 'SPM_mgL_std', 'SPM_mgL_p90',
    'B10_C_mean',
    'scene_cloud_pct', 'clear_water_frac',
    # Current weather and rolling weather history available on date T.
    'air_temp_c', 'wind_speed_10m_ms', 'precipitation_mm', 'solar_radiation_mj_m2',
    'air_temp_mean_prev_7d', 'lake_temp_mean_prev_7d',
    'precipitation_sum_prev_7d', 'wind_mean_prev_7d',
    'calm_days_prev_7d', 'solar_sum_prev_7d',
    'air_temp_mean_prev_14d', 'lake_temp_mean_prev_14d',
    'precipitation_sum_prev_14d', 'wind_mean_prev_14d',
    'season_sin', 'season_cos',
]
assert not any(name.startswith('cv_') or name.startswith('bloom_') for name in original_only_features)
assert set(original_only_features).issubset(cv_data.columns)

original_missingness = (cv_data[original_only_features].isna().mean()
    .sort_values(ascending=False).rename('missing_share').reset_index()
    .rename(columns={'index': 'feature'}))
print('Original-only numerical features:', len(original_only_features))
display(original_missingness.head(10).style.format({'missing_share': '{:.1%}'}))

In [ ]:
original_only_estimators = {
    'logistic regression': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'random forest': RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_leaf=3,
        class_weight='balanced_subsample', random_state=42, n_jobs=-1
    ),
    'histogram gradient boosting': HistGradientBoostingClassifier(
        max_iter=100, learning_rate=0.05, max_leaf_nodes=8,
        min_samples_leaf=8, l2_regularization=5.0, random_state=42
    ),
    'classic gradient boosting': GradientBoostingClassifier(
        n_estimators=75, learning_rate=0.03, max_depth=2,
        min_samples_leaf=6, random_state=42
    ),
}

original_only_oof = cv_data[['date', 'sensor', 'bloom_target_date_7d_050',
                             'bloom_target_delta_days_050', 'bloom_in_7d_cv_050', 'target_group']].copy()
for model_name in original_only_estimators:
    original_only_oof['p_' + model_name.replace(' ', '_')] = np.nan

for fold, (train_index, test_index) in enumerate(splitter.split(cv_data, y_cv, groups_cv), start=1):
    fold_train = cv_data.iloc[train_index]
    fold_test = cv_data.iloc[test_index]
    X_fold_train = fold_train[original_only_features + categorical_columns]
    X_fold_test = fold_test[original_only_features + categorical_columns]
    y_fold_train = fold_train['bloom_in_7d_cv_050']

    for model_name, estimator in original_only_estimators.items():
        model = Pipeline([('preprocessor', ColumnTransformer([
            ('numeric', Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
            ]), original_only_features),
            ('sensor', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_columns),
        ], sparse_threshold=0)), ('model', estimator)])
        model.fit(X_fold_train, y_fold_train)
        original_only_oof.loc[fold_test.index, 'p_' + model_name.replace(' ', '_')] = (
            model.predict_proba(X_fold_test)[:, 1]
        )

assert original_only_oof.filter(like='p_').notna().all().all()

In [ ]:
original_only_scores = []
for column in [column for column in original_only_oof if column.startswith('p_')]:
    probability = original_only_oof[column]
    original_only_scores.append({
        'model': column.removeprefix('p_').replace('_', ' '),
        'roc_auc': roc_auc_score(y_cv, probability),
        'brier_score': brier_score_loss(y_cv, probability),
        'pr_auc': average_precision_score(y_cv, probability),
        'mean_predicted_probability': probability.mean(),
    })
original_only_score_table = (pd.DataFrame(original_only_scores)
    .sort_values(['brier_score', 'roc_auc'], ascending=[True, False])
    .set_index('model'))
display(original_only_score_table.round(3))

# Exact paired comparison: same rows, same target groups and same folds.
comparison_models = ['logistic regression', 'random forest',
                     'histogram gradient boosting', 'classic gradient boosting']
ablation_comparison = pd.DataFrame(index=comparison_models)
for metric in ['roc_auc', 'brier_score', 'pr_auc']:
    ablation_comparison[f'with_CV_{metric}'] = oof_score_table.loc[comparison_models, metric]
    ablation_comparison[f'without_CV_{metric}'] = original_only_score_table.loc[comparison_models, metric]
ablation_comparison['CV_effect_on_AUC'] = (
    ablation_comparison['with_CV_roc_auc'] - ablation_comparison['without_CV_roc_auc']
)
ablation_comparison['CV_effect_on_Brier'] = (
    ablation_comparison['without_CV_brier_score'] - ablation_comparison['with_CV_brier_score']
)
display(ablation_comparison.round(3))
print('Positive CV_effect_on_AUC and CV_effect_on_Brier means that adding CV improved the model.')

In [ ]:
ORIGINAL_ONLY_OOF_FILE = ROOT / 'shershnevskoe_7d_probability_oof_original_only_relaxed_050.csv'
original_only_save = original_only_oof.sort_values(['date', 'sensor']).copy()
original_only_save['date'] = original_only_save['date'].dt.strftime('%Y-%m-%d')
original_only_save['bloom_target_date_7d_050'] = original_only_save['bloom_target_date_7d_050'].dt.strftime('%Y-%m-%d')
original_only_save.to_csv(ORIGINAL_ONLY_OOF_FILE, index=False, encoding='utf-8-sig')
print(f'Saved original-only out-of-fold probabilities: {ORIGINAL_ONLY_OOF_FILE.resolve()}')

## 12. Важность признаков: финальный Random Forest с CV

Финальный Random Forest обучается на всех 94 доступных строках
relaxed-выборки. Важность признака — это доля того, насколько он
помогал деревьям уменьшать неопределённость при разбиениях.

Это **не причинный вывод**: если признаки сильно похожи между собой,
модель может разделить их важность. В частности, текущий CV-сигнал и
FAI могут содержать пересекающуюся информацию.

In [ ]:
final_rf = Pipeline([
    ('preprocessor', make_preprocessor()),
    ('model', RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_leaf=3,
        class_weight='balanced_subsample', random_state=42, n_jobs=-1
    )),
])
X_final = cv_data[feature_columns + categorical_columns]
y_final = cv_data['bloom_in_7d_cv_050']
final_rf.fit(X_final, y_final)

transformed_feature_names = final_rf.named_steps['preprocessor'].get_feature_names_out()
rf_feature_importance = pd.DataFrame({
    'feature': transformed_feature_names,
    'importance': final_rf.named_steps['model'].feature_importances_,
})
rf_feature_importance['feature'] = (rf_feature_importance['feature']
    .str.replace('numeric__', '', regex=False)
    .str.replace('sensor__sensor_', 'sensor = ', regex=False))
rf_feature_importance['feature_group'] = np.where(
    rf_feature_importance['feature'].str.startswith('cv_'), 'CV current-state feature',
    np.where(rf_feature_importance['feature'].str.startswith('sensor = '), 'satellite type',
             'satellite / weather / season')
)
rf_feature_importance = rf_feature_importance.sort_values('importance', ascending=False).reset_index(drop=True)
display(rf_feature_importance)
print('Importance sum:', round(rf_feature_importance['importance'].sum(), 6))

In [ ]:
plot_data = rf_feature_importance.head(18).sort_values('importance')
colors = np.where(plot_data['feature_group'].eq('CV current-state feature'), '#2a9d8f',
         np.where(plot_data['feature_group'].eq('satellite type'), '#e9c46a', '#457b9d'))
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(plot_data['feature'], plot_data['importance'], color=colors)
ax.set(title='Random Forest feature importance: model with CV',
       xlabel='impurity-based importance (relative share)', ylabel='')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#2a9d8f', label='CV current-state feature'),
    Patch(color='#457b9d', label='satellite / weather / season'),
    Patch(color='#e9c46a', label='satellite type'),
], loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
RF_IMPORTANCE_FILE = ROOT / 'random_forest_feature_importance_with_cv_relaxed_050.csv'
rf_feature_importance.to_csv(RF_IMPORTANCE_FILE, index=False, encoding='utf-8-sig')
print(f'Saved feature importance: {RF_IMPORTANCE_FILE.resolve()}')

## 13. Показательный прогноз на каждый день сезона 2026 без CV

Этот блок показывает, как применять Histogram Gradient Boosting,
если на дату прогноза мы не запускаем CV. Модель обучается на
исторических целевых примерах с датой до 2026 года. В её входах
остаются только исходные спутниковые показатели, погодные данные,
сезон и тип спутника.

Для ежедневного графика погодные признаки берутся на каждый день.
Спутниковые признаки последнего пригодного снимка переносятся вперёд
максимум на 7 дней. Если более свежего пригодного снимка нет, на
графике будет разрыв. Поэтому линия не является ежедневным спутниковым
наблюдением: это ежедневный прогноз с последним доступным состоянием
воды.

In [ ]:
DEMO_YEAR = 2026
MAX_SATELLITE_AGE_DAYS = 7
DEMO_WEATHER_FILE = ROOT / 'shershnevskoe_daily_weather_era5land.csv'
DEMO_PLOT_FILE = ROOT / 'shershnevskoe_2026_daily_probability_without_cv.png'
DEMO_CSV_FILE = ROOT / 'shershnevskoe_2026_daily_probability_without_cv.csv'

# Train only on examples whose forecast targets were observable before 2026.
demo_train = labeled.loc[
    labeled['bloom_in_7d_cv_050'].notna() &
    (labeled['date'] < pd.Timestamp(f'{DEMO_YEAR}-01-01')) &
    (labeled['bloom_target_date_7d_050'] < pd.Timestamp(f'{DEMO_YEAR}-01-01'))
].copy()
demo_train['bloom_in_7d_cv_050'] = demo_train['bloom_in_7d_cv_050'].astype(int)

# Load daily weather and retain only the chosen year.
daily_weather = pd.read_csv(DEMO_WEATHER_FILE, parse_dates=['date'])
daily_weather = daily_weather.loc[daily_weather['date'].dt.year.eq(DEMO_YEAR)].copy()
daily_weather['season_sin'] = np.sin(2 * np.pi * daily_weather['date'].dt.dayofyear / 365.25)
daily_weather['season_cos'] = np.cos(2 * np.pi * daily_weather['date'].dt.dayofyear / 365.25)

# Use only the original satellite aggregate, not any cv_* column.
satellite_features = [
    'FAI_mean', 'FAI_median', 'FAI_std', 'FAI_p90',
    'MNDWI_mean', 'MNDWI_std', 'MNDWI_p90',
    'Turbidity_FNU_mean', 'Turbidity_FNU_std', 'Turbidity_FNU_p90',
    'SPM_mgL_mean', 'SPM_mgL_std', 'SPM_mgL_p90', 'B10_C_mean',
    'scene_cloud_pct', 'clear_water_frac',
]
satellite_history = merged.loc[
    merged['date'].dt.year.eq(DEMO_YEAR) & merged['usable_for_fai'].fillna(False),
    ['date', 'sensor', *satellite_features]
].copy()
satellite_history = (satellite_history
    .sort_values(['date', 'clear_water_frac'])
    .drop_duplicates('date', keep='last')
    .rename(columns={'date': 'satellite_observation_date'}))

demo_daily = pd.merge_asof(
    daily_weather.sort_values('date'),
    satellite_history.sort_values('satellite_observation_date'),
    left_on='date', right_on='satellite_observation_date', direction='backward'
)
demo_daily['satellite_age_days'] = (
    demo_daily['date'] - demo_daily['satellite_observation_date']
).dt.days
demo_daily = demo_daily.loc[
    demo_daily['satellite_age_days'].between(0, MAX_SATELLITE_AGE_DAYS)
].copy()

# This model's inputs are original satellite + weather + season + sensor only.
demo_feature_columns = original_only_features
assert not any(name.startswith('cv_') or name.startswith('bloom_') for name in demo_feature_columns)
assert set(demo_feature_columns + categorical_columns).issubset(demo_daily.columns)

demo_model = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), demo_feature_columns),
        ('sensor', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_columns),
    ], sparse_threshold=0)),
    ('model', HistGradientBoostingClassifier(
        max_iter=100, learning_rate=0.05, max_leaf_nodes=8,
        min_samples_leaf=8, l2_regularization=5.0, random_state=42
    )),
])
X_demo_train = demo_train[demo_feature_columns + categorical_columns]
y_demo_train = demo_train['bloom_in_7d_cv_050']
demo_model.fit(X_demo_train, y_demo_train)
demo_daily['probability_bloom_in_5_9_days'] = demo_model.predict_proba(
    demo_daily[demo_feature_columns + categorical_columns]
)[:, 1]

print(f'Training examples before {DEMO_YEAR}: {len(demo_train)}; blooms={y_demo_train.sum()}; no-bloom={len(y_demo_train) - y_demo_train.sum()}')
print(f'Daily weather rows available in {DEMO_YEAR}: {len(daily_weather)}')
print(f'Days with a usable satellite image no older than {MAX_SATELLITE_AGE_DAYS} days: {len(demo_daily)}')
print(f'Period shown: {demo_daily.date.min().date()} — {demo_daily.date.max().date()}')
print('No CV field is used in demo_model input. Historical labels were generated from the provisional CV workflow.')

In [ ]:
# Keep the full daily calendar so periods without a fresh image show as gaps.
demo_plot_data = daily_weather[['date']].merge(
    demo_daily[['date', 'probability_bloom_in_5_9_days', 'satellite_observation_date',
                'satellite_age_days', 'sensor']],
    on='date', how='left'
)
acquisition_days = demo_plot_data['satellite_age_days'].eq(0)

fig, ax = plt.subplots(figsize=(14, 5.5))
ax.plot(demo_plot_data['date'], demo_plot_data['probability_bloom_in_5_9_days'],
        color='#1f4e78', linewidth=1.8, label='Вероятность цветения через 5–9 дней')
ax.scatter(demo_plot_data.loc[acquisition_days, 'date'],
           demo_plot_data.loc[acquisition_days, 'probability_bloom_in_5_9_days'],
           color='#e76f51', edgecolor='white', linewidth=0.6, s=38,
           zorder=3, label='Дата нового пригодного спутникового снимка')
ax.set_ylim(0, 1)
ax.set_ylabel('Прогнозируемая вероятность')
ax.set_xlabel('Дата прогноза в 2026 году')
ax.set_title('Показательный прогноз цветения на 2026 год без CV-признаков')
ax.grid(axis='y', alpha=0.25)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.15), ncol=2, frameon=False)
ax.text(0.01, -0.20,
        'На каждый день используется погода этого дня и последний пригодный снимок (не старше 7 дней). '
        'Пробелы означают, что свежего пригодного спутникового снимка нет. '
        '2026 год пока неполный; прогнозы показательные, не подтверждённые независимыми биологическими измерениями.',
        transform=ax.transAxes, fontsize=9, va='top', wrap=True)
plt.tight_layout()
plt.savefig(DEMO_PLOT_FILE, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved annual probability plot: {DEMO_PLOT_FILE.resolve()}')

In [ ]:
demo_output_columns = [
    'date', 'probability_bloom_in_5_9_days', 'satellite_observation_date',
    'satellite_age_days', 'sensor'
]
demo_daily_output = demo_plot_data[demo_output_columns].copy()
for col in ['date', 'satellite_observation_date']:
    demo_daily_output[col] = demo_daily_output[col].dt.strftime('%Y-%m-%d')
demo_daily_output.to_csv(DEMO_CSV_FILE, index=False, encoding='utf-8-sig')
display(demo_daily_output.dropna(subset=['probability_bloom_in_5_9_days']).head(12))
print(f'Saved daily probabilities: {DEMO_CSV_FILE.resolve()}')

# As an honest diagnostic, count available 2026 target labels.
demo_2026_labels = labeled.loc[
    labeled['date'].dt.year.eq(DEMO_YEAR) & labeled['bloom_in_7d_cv_050'].notna()
].copy()
demo_2026_labels['bloom_in_7d_cv_050'] = demo_2026_labels['bloom_in_7d_cv_050'].astype(int)
if len(demo_2026_labels):
    demo_2026_labels['p_original_only_model'] = demo_model.predict_proba(
        demo_2026_labels[demo_feature_columns + categorical_columns]
    )[:, 1]
    demo_2026_brier = brier_score_loss(
        demo_2026_labels['bloom_in_7d_cv_050'],
        demo_2026_labels['p_original_only_model']
    )
    print(f'CV-derived retrospective 7-day labels in 2026: {len(demo_2026_labels)}; blooms={demo_2026_labels.bloom_in_7d_cv_050.sum()}')
    print(f'Mean forecast probability on these labelled dates: {demo_2026_labels.p_original_only_model.mean():.3f}; Brier score={demo_2026_brier:.3f}')
    if demo_2026_labels['bloom_in_7d_cv_050'].nunique() < 2:
        print('ROC-AUC is undefined because 2026 labels contain only one class.')
print('These labels come from the CV pipeline and are too few and not biologically verified. Treat this as a warning signal, not a reliable accuracy estimate.')

## 14. Сохранение объединённой и размеченной базы

In [ ]:
output = merged.copy()
output['date'] = output['date'].dt.strftime('%Y-%m-%d')
output.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Saved: {OUTPUT_FILE.resolve()}')
print(f'Rows: {len(output)}; columns: {len(output.columns)}')

labeled_output = labeled.copy()
labeled_output['date'] = labeled_output['date'].dt.strftime('%Y-%m-%d')
labeled_output['bloom_target_date_7d'] = labeled_output['bloom_target_date_7d'].dt.strftime('%Y-%m-%d')
labeled_output['bloom_target_date_7d_050'] = labeled_output['bloom_target_date_7d_050'].dt.strftime('%Y-%m-%d')
labeled_output.to_csv(LABELED_OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Saved labelled dataset: {LABELED_OUTPUT_FILE.resolve()}')

## 15. Walk-forward-проверка по годам без CV-признаков

Расширяющееся окно: каждый тестовый год прогнозируется только по более ранним обучающим данным. Для обучающих меток дата будущей CV-сцены также должна быть до начала тестового года. Ромбы показывают CV-метку (0/1) для пригодной сцены через 5–9 дней; это прокси-разметка, не независимое биологическое наблюдение.

## Что можно использовать дальше

Для временной прогнозной модели используем CV-поля только как
признаки текущей даты: `cv_mean_bloom_probability`,
`cv_bloom_fraction_non_artifact`, `cv_artifact_fraction`.
Строки с `cv_usable_strict = False` не должны подменять отсутствие
цветения нулём: это означает плохое качество снимка, а не отсутствие
цветения.

In [ ]:
# Expanding-window evaluation by held-out year; no future labels enter training.
WF_PLOT_FILE = ROOT / 'shershnevskoe_walk_forward_cv_comparison.png'
WF_CSV_FILE = ROOT / 'shershnevskoe_walk_forward_cv_comparison.csv'
wf_rows = cv_data.dropna(subset=['bloom_in_7d_cv_050', 'bloom_target_date_7d_050']).copy()
wf_rows['bloom_in_7d_cv_050'] = wf_rows['bloom_in_7d_cv_050'].astype(int)
wf_predictions = []
wf_years = sorted(y for y in wf_rows['date'].dt.year.unique() if y >= 2024)
for test_year in wf_years:
    cutoff = pd.Timestamp(f'{test_year}-01-01')
    wf_train = wf_rows.loc[(wf_rows['date'] < cutoff) & (wf_rows['bloom_target_date_7d_050'] < cutoff)].copy()
    wf_test = wf_rows.loc[wf_rows['date'].dt.year.eq(test_year)].copy()
    if wf_train['bloom_in_7d_cv_050'].nunique() < 2 or wf_test.empty:
        print(f'Skip {test_year}: training needs both classes and test must be non-empty')
        continue
    wf_model = Pipeline([('preprocessor', ColumnTransformer([
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), original_only_features),
        ('sensor', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_columns),
    ], sparse_threshold=0)), ('model', HistGradientBoostingClassifier(
        max_iter=100, learning_rate=0.05, max_leaf_nodes=8, min_samples_leaf=8,
        l2_regularization=5.0, random_state=42
    ))])
    wf_model.fit(wf_train[original_only_features + categorical_columns], wf_train['bloom_in_7d_cv_050'])
    wf_test['p_model'] = wf_model.predict_proba(wf_test[original_only_features + categorical_columns])[:, list(wf_model.classes_).index(1)]
    wf_test['test_year'] = test_year
    wf_predictions.append(wf_test[['date', 'sensor', 'bloom_target_date_7d_050', 'bloom_in_7d_cv_050', 'p_model', 'test_year']])

if not wf_predictions:
    raise ValueError('No walk-forward folds could be fitted')
wf_oof = pd.concat(wf_predictions, ignore_index=True).sort_values('date')
wf_scores = (wf_oof.groupby('test_year').apply(lambda g: pd.Series({
    'n_test': len(g), 'cv_blooms': int(g['bloom_in_7d_cv_050'].sum()),
    'mean_predicted_probability': g['p_model'].mean(),
    'brier_score': brier_score_loss(g['bloom_in_7d_cv_050'], g['p_model']),
    'roc_auc': roc_auc_score(g['bloom_in_7d_cv_050'], g['p_model']) if g['bloom_in_7d_cv_050'].nunique() == 2 else np.nan,
}), include_groups=False).round(3))
display(wf_scores)
wf_scores.to_csv(ROOT / 'shershnevskoe_walk_forward_metrics.csv', encoding='utf-8-sig')

fig, ax = plt.subplots(figsize=(14, 6))
ax.scatter(wf_oof['date'], wf_oof['p_model'], s=42, color='#1f4e78', label='Модель: вероятность цветения через 5–9 дней', zorder=3)
ax.scatter(wf_oof['date'], wf_oof['bloom_in_7d_cv_050'], s=52, marker='D', facecolors='none', edgecolors='#e76f51', linewidths=1.5, label='CV-метка будущей сцены: 0/1', zorder=4)
for year in wf_years[1:]:
    ax.axvline(pd.Timestamp(f'{year}-01-01'), color='gray', alpha=0.28, linestyle='--')
ax.set_ylim(-0.04, 1.04)
ax.set_ylabel('Вероятность модели / CV-метка (0 или 1)')
ax.set_xlabel('Дата выпуска прогноза; CV проверяет сцену через 5–9 дней')
ax.set_title('Walk-forward: прогноз модели против CV-разметки')
ax.grid(axis='y', alpha=0.25)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False)
ax.text(0.01, -0.20, 'Обучение только на более ранних годах. CV-метка предварительная и не является независимым биологическим измерением.', transform=ax.transAxes, fontsize=9, va='top', wrap=True)
plt.tight_layout()
plt.savefig(WF_PLOT_FILE, dpi=200, bbox_inches='tight')
plt.show()
wf_oof.to_csv(WF_CSV_FILE, index=False, encoding='utf-8-sig')
print(f'Saved walk-forward plot: {WF_PLOT_FILE.resolve()}')
print(f'Saved walk-forward predictions: {WF_CSV_FILE.resolve()}')